Skrypt demonstruje użycie środowiska LangFuse do ewaluacji i śledzenia pracy rozwiązań agentowych na przykładzie OpenAI Agents.

# Setup

In [ ]:
!uv pip install -q openai-agents nest_asyncio pydantic-ai[logfire] langfuse datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.9/161.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.4/275.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.5/194.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.0/264.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.5/127.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

*   **openai-agents:** Biblioteka od OpenAI, która umożliwia tworzenie agentów wykorzystujących modele językowe (np. GPT) do wykonywania zadań.
*   **nest_asyncio:** Pozwala na uruchamianie zagnieżdżonych pętli zdarzeń asynchronicznych w środowiskach, które już mają działającą pętlę zdarzeń (np. Jupyter Notebook). Jest to przydatne przy pracy z bibliotekami asynchronicznymi takimi jak `openai-agents`.
*   **pydantic-ai[logfire]:** Rozszerzenie dla biblioteki Pydantic, która służy do walidacji danych i tworzenia ustawień. Dodatkowa część `[logfire]` instaluje narzędzie LogFire, które ułatwia tworzenie interfejsów wiersza poleceń (CLI) z definicji modeli Pydantic.
*   **langfuse:** Platforma do monitorowania, debugowania i oceny aplikacji opartych na modelach językowych. Umożliwia śledzenie interakcji agentów, logowanie danych i analizę wyników.
*   **datasets:** Biblioteka od Hugging Face, która zapewnia łatwy dostęp do szerokiej gamy zbiorów danych (datasetów) używanych w uczeniu maszynowym i przetwarzaniu języka naturalnego.

In [ ]:
# Standard library
import asyncio
import base64
import os

# Third-party libraries
import nest_asyncio
import logfire

# OpenTelemetry
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry import trace

# Data & ML

# Local modules
from agents import Agent, Runner, WebSearchTool, function_tool


# Enable nested asyncio loops
nest_asyncio.apply()


from google.colab import userdata

**OpenTelemetry:**

*   Zawiera moduły związane z OpenTelemetry, systemem monitorowania i śledzenia rozproszonych aplikacji.  Konkretnie importowane są klasy do eksportu danych śledzenia (OTLPSpanExporter), konfiguracji dostawcy śledzenia (TracerProvider) oraz procesora spanów (SimpleSpanProcessor). Dodatkowo importowany jest moduł `trace` i funkcja `format_trace_id`.

**Dane i uczenie maszynowe:**

*   `datasets`: Biblioteka do łatwego pobierania i pracy ze zbiorami danych.
*   `langfuse`: Platforma do monitorowania, debugowania i oceny aplikacji opartych na modelach językowych. Importowane są zarówno główne klasy `Langfuse`, jak i sam moduł `langfuse`.

**Moduły lokalne:**

*   `agents`: Zawiera definicje klas takich jak `Agent`, `Runner`, `WebSearchTool` oraz `function_tool`. Te klasy implementują logikę agenta, jego uruchamiania, narzędzia do wyszukiwania w sieci i funkcjonalności oparte na funkcjach.

In [ ]:
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("langfuse_pub")
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("langfuse_prv")
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"  # 🇪🇺 EU region

LANGFUSE_AUTH = base64.b64encode(
    f"{os.environ.get('LANGFUSE_PUBLIC_KEY')}:{os.environ.get('LANGFUSE_SECRET_KEY')}".encode()
).decode()

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = (
    os.environ.get("LANGFUSE_HOST") + "/api/public/otel"
)
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"

os.environ["OPENAI_API_KEY"] = userdata.get("openaivision")

In [ ]:
# Create a TracerProvider for OpenTelemetry
trace_provider = TracerProvider()

# Add a SimpleSpanProcessor with the OTLPSpanExporter to send traces
trace_provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter()))

trace.set_tracer_provider(trace_provider)
tracer = trace.get_tracer(__name__)

Ten kod konfiguruje OpenTelemetry do śledzenia działania programu.

Najpierw tworzy instancję `TracerProvider`, która jest centralnym punktem zarządzania śledzeniem w OpenTelemetry.  `TracerProvider` odpowiada za tworzenie i zarządzanie tracerami.

Następnie dodaje do `TracerProvider` procesor spanów (`SimpleSpanProcessor`) wraz z eksporterem OTLP (`OTLPSpanExporter`). Procesor spanów przetwarza spany (jednostki śledzenia) generowane przez program, a eksporter OTLP wysyła te spany do punktu końcowego OpenTelemetry, który został wcześniej skonfigurowany w zmiennej środowiskowej `OTEL_EXPORTER_OTLP_ENDPOINT` (w poprzednim fragmencie kodu ustawiono go na serwer Langfuse).

Następnie kod ustawia globalnego domyślnego dostawcę śledzenia (`trace.set_tracer_provider`) na utworzony wcześniej `TracerProvider`.  Dzięki temu wszystkie wywołania funkcji śledzących w programie będą korzystały z tej konfiguracji.

Na koniec pobiera tracer o nazwie odpowiadającej aktualnemu modułowi (`__name__`). Tracer jest używany do tworzenia spanów, które reprezentują poszczególne operacje wykonywane przez program.  Ten tracer będzie teraz rejestrował spany i wysyłał je do Langfuse w celu monitorowania i analizy.

In [ ]:
# Configure logfire instrumentation.
logfire.configure(
    service_name="my_agent_service",
    send_to_logfire=False,
)
# This method automatically patches the OpenAI Agents SDK to send logs via OTLP to Langfuse.
logfire.instrument_openai_agents()

Ten kod konfiguruje integrację biblioteki `logfire` z aplikacją i automatycznie instrumentuje bibliotekę OpenAI Agents do wysyłania logów za pomocą OpenTelemetry (OTLP) do platformy Langfuse.

Najpierw wywołuje funkcję `logfire.configure()`, która ustawia parametry konfiguracji dla `logfire`.  Ustawiono nazwę usługi (`service_name`) na 'my\_agent\_service'. Flaga `send_to_logfire=False` oznacza, że logi nie będą wysyłane bezpośrednio do serwerów Logfire (prawdopodobnie dlatego, że dane są wysyłane przez OTLP do Langfuse).

Następnie wywołuje funkcję `logfire.instrument_openai_agents()`. Ta funkcja automatycznie modyfikuje kod biblioteki OpenAI Agents tak, aby logi generowane przez tę bibliotekę były przesyłane za pomocą protokołu OpenTelemetry (OTLP) do skonfigurowanego wcześniej punktu końcowego Langfuse.  Dzięki temu wszystkie logi z agentów OpenAI będą śledzone i dostępne w interfejsie Langfuse, co ułatwia debugowanie i monitorowanie działania aplikacji.

# Test

In [ ]:
async def main():
    agent = Agent(
        name="Assistant",
        instructions="You are a senior software engineer",
    )

    result = await Runner.run(
        agent, "Tell me why it is important to evaluate AI agents."
    )
    print(result.final_output)


loop = asyncio.get_running_loop()
await loop.create_task(main())


11:58:52.912 OpenAI Agents trace: Agent workflow
11:58:52.944   Agent run: 'Assistant'
11:58:53.556     Responses API with 'gpt-4o'
Evaluating AI agents is crucial for several reasons:

1. **Performance Measurement**: It helps determine how well an AI agent performs its intended tasks compared to benchmarks or human-level competence.

2. **Reliability and Safety**: Evaluation ensures that the AI operates safely, especially in critical applications like healthcare or autonomous driving, where errors can have serious consequences.

3. **Transparency and Trust**: By assessing AI agents, developers and users gain insights into their decision-making processes, building trust and transparency.

4. **Bias and Fairness**: Evaluation can uncover biases in AI systems, ensuring that they make fair and equitable decisions across different groups and scenarios.

5. **Continuous Improvement**: Regular assessment provides feedback that can be used to refine and improve AI models over time.

6. **Comp

Ten kod definiuje i uruchamia główną funkcję asynchroniczną `main()`, która tworzy agenta i wykonuje z nim zadanie, a następnie wyświetla wynik.

Funkcja `main()` najpierw tworzy instancję klasy `Agent` o nazwie "Assistant" i przekazuje mu instrukcje: "You are a senior software engineer".  Instrukcje te definiują rolę i zachowanie agenta.

Następnie używa klasy `Runner` do uruchomienia agenta z podanym zadaniem: "Tell me why it is important to evaluate AI agents.". Funkcja `Runner.run()` jest asynchroniczna, więc używany jest operator `await`, aby poczekać na zakończenie działania agenta i uzyskanie wyniku.

Wynik działania agenta (zawierający odpowiedź na pytanie) jest przechowywany w zmiennej `result`. Następnie wyświetla się końcowy wynik (`result.final_output`) za pomocą funkcji `print()`.

Na końcu kodu pobiera się aktualną pętlę zdarzeń asyncio (`asyncio.get_running_loop()`) i tworzy zadanie asynchroniczne (`loop.create_task(main())`) do uruchomienia funkcji `main()`.  Uruchomienie zadania w ten sposób pozwala na wykonanie funkcji `main()` w tle, bez blokowania głównego wątku programu.

Wyniki: https://cloud.langfuse.com/project/cmam55kmo0011ad073ozxrz6h/traces


# Test 2

In [ ]:
@function_tool
def get_weather(city: str) -> str:
    return f"The weather in {city} is sunny."


agent = Agent(
    name="Hello world",
    instructions="You are a helpful agent.",
    tools=[get_weather],
)


async def main():
    result = await Runner.run(agent, input="What's the weather in Berlin?")
    print(result.final_output)


loop = asyncio.get_running_loop()
await loop.create_task(main())

12:20:03.516 OpenAI Agents trace: Agent workflow
12:20:03.518   Agent run: 'Hello world'
12:20:03.520     Responses API with 'gpt-4o'
12:20:04.464     Function: get_weather
12:20:04.468     Responses API with 'gpt-4o'
The weather in Berlin is currently sunny.


Ten kod definiuje narzędzie (funkcję) do pobierania informacji o pogodzie i wykorzystuje je w agencie, który odpowiada na pytanie użytkownika dotyczące pogody w Berlinie.

Najpierw dekorator `@function_tool` jest używany do oznaczenia funkcji `get_weather` jako narzędzia dostępnego dla agenta. Funkcja ta przyjmuje nazwę miasta (`city`) jako argument i zwraca tekst informujący, że pogoda w tym mieście jest słoneczna.  W rzeczywistej implementacji funkcja ta prawdopodobnie pobierałaby dane pogodowe z zewnętrznego API.

Następnie tworzona jest instancja klasy `Agent` o nazwie "Hello world" i przekazywane są mu instrukcje: "You are a helpful agent.". Lista narzędzi (`tools`) zawiera funkcję `get_weather`, co oznacza, że agent może używać tej funkcji do wykonywania zadań.

Funkcja asynchroniczna `main()` uruchamia agenta za pomocą klasy `Runner` z zapytaniem użytkownika: "What's the weather in Berlin?".  Agent analizuje pytanie i, ponieważ ma dostęp do narzędzia `get_weather`, wywołuje tę funkcję z argumentem "Berlin".

Wynik działania agenta (zawierający odpowiedź na pytanie) jest przechowywany w zmiennej `result`. Następnie wyświetla się końcowy wynik (`result.final_output`) za pomocą funkcji `print()`. W tym przypadku, ponieważ funkcja `get_weather` zawsze zwraca informację o słonecznej pogodzie, program wypisze: "The weather in Berlin is sunny.".

Na końcu kodu pobiera się aktualną pętlę zdarzeń asyncio i tworzy zadanie asynchroniczne do uruchomienia funkcji `main()`.

Wyniki: https://cloud.langfuse.com/project/cmam55kmo0011ad073ozxrz6h/traces

# Test 3

In [ ]:
input_query = "Why is AI agent evaluation important?"

with tracer.start_as_current_span("OpenAI-Agent-Trace") as span:
    span.set_attribute("langfuse.user.id", "user-12345")
    span.set_attribute("langfuse.session.id", "my-agent-session")
    span.set_attribute("langfuse.tags", ["staging", "demo", "OpenAI Agent SDK"])

    async def main(input_query):
        agent = Agent(
            name="Assistant",
            instructions="You are a helpful assistant.",
        )

        result = await Runner.run(agent, input_query)
        print(result.final_output)
        return result

    result = await main(input_query)

    # Add input and output values to parent trace
    span.set_attribute("input.value", input_query)
    span.set_attribute("output.value", result.final_output)

12:31:11.099 OpenAI Agents trace: Agent workflow
12:31:11.102   Agent run: 'Assistant'
12:31:11.106     Responses API with 'gpt-4o'
AI agent evaluation is crucial for several reasons:

1. **Performance Measurement**: It helps assess how well an AI agent performs in its intended tasks, ensuring it meets specified objectives.

2. **Safety and Reliability**: Evaluation ensures that the agent operates safely and reliably, minimizing potential risks and unintended behaviors.

3. **Improvement and Optimization**: By identifying strengths and weaknesses, evaluation aids in refining and optimizing AI models for better performance.

4. **Bias Detection**: It helps reveal biases in AI behavior, enabling developers to address and mitigate these biases to create fairer systems.

5. **Accountability**: Evaluating AI systems holds creators accountable for the functionality and impact of their agents.

6. **Compliance and Standards**: It ensures that AI systems meet industry standards and regulatory 

Ten kod uruchamia agenta AI z określonym zapytaniem, śledzi jego działanie za pomocą OpenTelemetry i dodaje metadane do śladu w celu integracji z Langfuse.

Najpierw definiowana jest zmienna `input_query` zawierająca pytanie dla agenta: "Why is AI agent evaluation important?".

Następnie używany jest kontekst menedżera `with tracer.start_as_current_span("OpenAI-Agent-Trace") as span:` do rozpoczęcia śledzenia działania agenta.  Funkcja `tracer.start_as_current_span()` tworzy nowy span o nazwie "OpenAI-Agent-Trace" i ustawia go jako aktywny span w bieżącym kontekście.

Wewnątrz bloku `with` dodawane są atrybuty do spanu, które zawierają metadane dotyczące użytkownika i sesji:
*   `langfuse.user.id`: Identyfikator użytkownika ("user-12345").
*   `langfuse.session.id`: Identyfikator sesji ("my-agent-session").
*   `langfuse.tags`: Lista tagów opisujących kontekst działania agenta (["staging", "demo", "OpenAI Agent SDK"]).

Następnie definiowana jest asynchroniczna funkcja `main(input_query)`, która tworzy agenta o nazwie "Assistant" z instrukcją: "You are a helpful assistant.".  Funkcja uruchamia agenta za pomocą klasy `Runner` z podanym zapytaniem (`input_query`) i wyświetla końcowy wynik (`result.final_output`). Funkcja zwraca również obiekt `result`.

Wynik działania funkcji `main()` jest przechowywany w zmiennej `result`.

Po zakończeniu działania agenta, do spanu dodawane są atrybuty zawierające wartości wejściowe i wyjściowe:
*   `input.value`: Zapytanie użytkownika (`input_query`).
*   `output.value`: Końcowy wynik zwrócony przez agenta (`result.final_output`).

Dzięki temu, cały proces działania agenta (od otrzymania zapytania do wygenerowania odpowiedzi) jest śledzony za pomocą OpenTelemetry, a metadane dotyczące użytkownika, sesji i wartości wejściowych/wyjściowych są dodawane do spanu w celu integracji z platformą Langfuse.  Langfuse może wykorzystać te dane do monitorowania wydajności agenta, debugowania problemów i analizy zachowań.

# Test 4

Tworzymy LLM do oceny:

Evaluation -> LLM-as-a-judge -> create evaluator -> create new template

Evaluation -> create evaluator

In [ ]:
# Define your agent with the web search tool
agent = Agent(
    name="WebSearchAgent",
    instructions="You are an agent that can search the web.",
    tools=[WebSearchTool()],
)

input_query = "Is eating carrots good for the eyes?"

# Run agent
with trace.get_tracer(__name__).start_as_current_span("OpenAI-Agent-Trace") as span:
    # Run your agent with a query
    result = Runner.run_sync(agent, input_query)

    # Add input and output values to parent trace
    span.set_attribute("input.value", input_query)
    span.set_attribute("output.value", result.final_output)

14:27:48.134 OpenAI Agents trace: Agent workflow
14:27:48.135   Agent run: 'WebSearchAgent'
14:27:48.136     Responses API with 'gpt-4o'


Ten kod definiuje agenta AI z narzędziem do wyszukiwania w sieci i uruchamia go z określonym zapytaniem, śledząc jego działanie za pomocą OpenTelemetry.

Najpierw tworzona jest instancja klasy `Agent` o nazwie "WebSearchAgent" i przekazywane są mu instrukcje: "You are an agent that can search the web.". Lista narzędzi (`tools`) zawiera instancję klasy `WebSearchTool`, co oznacza, że agent może używać tego narzędzia do wyszukiwania informacji w sieci.

Następnie definiowana jest zmienna `input_query` zawierająca pytanie dla agenta: "Is eating carrots good for the eyes?".

Kod uruchamia agenta za pomocą kontekstu menedżera `with trace.get_tracer(__name__).start_as_current_span("OpenAI-Agent-Trace") as span:`.  Funkcja `trace.get_tracer(__name__).start_as_current_span()` tworzy nowy span o nazwie "OpenAI-Agent-Trace" i ustawia go jako aktywny span w bieżącym kontekście.

Wewnątrz bloku `with` uruchamiana jest funkcja `Runner.run_sync(agent, input_query)`. Funkcja ta uruchamia agenta synchronicznie z podanym zapytaniem i zwraca wynik.

Wynik działania agenta jest przechowywany w zmiennej `result`.

Po zakończeniu działania agenta, do spanu dodawane są atrybuty zawierające wartości wejściowe i wyjściowe:
*   `input.value`: Zapytanie użytkownika (`input_query`).
*   `output.value`: Końcowy wynik zwrócony przez agenta (`result.final_output`).

Dzięki temu, cały proces działania agenta (od otrzymania zapytania do wygenerowania odpowiedzi) jest śledzony za pomocą OpenTelemetry, a metadane dotyczące wartości wejściowych i wyjściowych są dodawane do spanu w celu monitorowania i analizy.  W tym przypadku agent użyje narzędzia `WebSearchTool` do znalezienia informacji na temat wpływu jedzenia marchewki na wzrok i wykorzysta te informacje do sformułowania odpowiedzi.